# LangChain: 챗봇 구현

## 환경 설정

In [1]:
# 필요한 모듈 / 함수 / 클래스 임포트
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

In [2]:
# .env file -> API Key 가져오기
env_file_path='.env'
output = load_dotenv(env_file_path)
print(output)

True


## LLM 모델 생성

In [3]:
# ChatOpenAI 클래스 호출, llm 모델 생성
llm = ChatOpenAI(model='gpt-5-nano')

## 대화형 프롬프트 정의 (MessagesPlaceholder 활용)

In [4]:
"""
# MessagesPlaceholder 클래스
1. 연속적인 대화가 가능하기 위해서는 LLM이 이전 대화를 기억하고 있어야 한다.
2. 기능 구현
   1) system 프롬프트 이후에 생성되는 사용자 요청, AI 답변(LLM 답변), 사용자 새로운 요청, AI 답변(LLM 답변)을
   chain 실행 전에 chat_history 이름의 리스트에 넣어서 전체 프롬프트에 입력한다.
   2) 전체 프롬프트에 chat_history라는 이름의 리스트가 들어갈 공간이 필요하고, 이를 제공하는 것이
   MessagesPlaceholder의 기능이다.
   3)모든 대화 이력이 포함된 프롬프트를 LLM에게 전달한다.
"""

# 프롬프트 생성
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 친절하고 전문적인 AI 비서야. 이전 대화 맥락을 정확하게 기억하여 사용자의 질문에 일관성 있게 답변해야 해"),
        MessagesPlaceholder(variable_name="chat_history")
    ]
)

print(prompt)

input_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messages.functi

In [5]:
# 출력 도구 생성 -> LLM 출력 분석 -> 텍스트 전달 
output_parser = StrOutputParser()

## Chain 생성

In [6]:
# LCEL 방식: '|' 기호로 연결
chatbot_chain = prompt | llm | output_parser
print(chatbot_chain)

first=ChatPromptTemplate(input_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[lang

## Chatbot 실행

In [7]:
# 대화 히스토리를 저장할 빈 리스트 생성
chat_history = []

print("대화형 AI 챗봇이 시작되었습니다!. \n")


# 종료 조건이 충족되기전까지 계속 실행 -> while 반복문
while True:
    # 사용자 입력 함수 호출 -> 사용자 입력 생성
    user_input = input("사용자 입력: ")

    # 종료 조건 설정: 사용자 입력이 ["q", "quit", "exit", "종료"]에 속하면 종료
    if user_input.lower() in ["q", "quit", "exit", "종료"]:
        break

    # chain을 호출하기 전, 사용자 질문(입력)을 chat_history에 추가한다.
    """
    # user_input -> 자료형: 문자열 -> LangChain 활용 -> HumanMessage 자료형
    """
    chat_history.append(HumanMessage(content=user_input))

    # invoke() 함수 호출, AI 답변 생성
    response = chatbot_chain.invoke({"chat_history":chat_history})
    print(f"AI 챗봇 답변: {response}\n")

    # LLM 답변 생성이 끝난 후, LLM의 답변을 AIMessage 클래스를 이용, chat_history 추가
    chat_history.append(AIMessage(content=response))

대화형 AI 챗봇이 시작되었습니다!. 

AI 챗봇 답변: 아래 세 가지를 초기 기획에서 가장 먼저 결정하면 좋아요.

1) 해결하려는 문제와 가치 제안(ROI) 정의
- 어떤 소상공인 문제를 해결하고 싶은지 명확히 하고, 기대 효과를 수치(KPI)로 설정합니다(예: 매출 증가, 비용 절감, 작업 시간 감소).

2) 대상 고객(세그먼트)과 핵심 사용 시나리오
- 타깃 업종/기업 규모/기술 수용도 등의 구체적 세그먼트를 정의하고, 가장 가치 있는 2–3가지 사용 사례와 해당 흐름을 명시합니다.

3) 데이터/기술 가능성 및 MVP 범위
- 필요한 데이터 소스, 접근성, 프라이버시/보안 이슈 파악. 기술 선택(모델 유형, 인프라)과 함께, 최소한의 기능으로 가치를 검증할 수 있는 MVP 범위를 정의합니다. 

필요하면 산업군이나 목표 예산을 알려주면, 이 세 가지를 바탕으로 맞춤형 방향도 같이 정리해 드릴게요.

AI 챗봇 답변: 좋습니다. 두 번째 요소인 대상 고객(세그먼트)와 핵심 시나리오를 실제로 구현하려면 어떤 기술 스택으로 접근하면 효율적인지 정리해 드릴게요. 초점은 세그먼트/시나리오를 정의하고 이를 관리하고 시각화하는 데 필요한 도구와 아키텍처입니다.

1) 노코드/저코드로 빠르게 시도하는 경로 (초기 실험용, 예산 적은 경우)
- 데이터 저장 및 관리
  - Airtable: Segment(세그먼트), Persona(페르소나), UseCase(핵심 시나리오), JourneyStep(여정 단계) 등을 하나의 데이터베이스처럼 관리
- UI/포털 구성
  - Softr, Stacker, Pory 등: Airtable 데이터를 바탕으로 내부 구성원용 포털이나 간단한 관리 화면 구축
- 문서화/정의 도구
  - Notion이나 Coda: 시나리오 맵, 의사결정 로그, KPI 정의를 중앙화
- 설계 도구
  - Figma: 페르소나와 여정 흐름, 와이어프레임 제작
- 자동화/연동
  - Zapier나 Make(과거 Integromat): Airtable과 No